In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io
import re

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 26
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## World Bank Carbon Pricing Dashboard Pipeline

**Source:** World Bank Carbon Pricing Dashboard
**Access:** Automated OWID CSV — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — WB_CARBON section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Carbon tax instrument coverage | Environmental/climate governance | Primary tier 1 |

In [2]:
import requests
import io
import pandas as pd
from datetime import datetime

# World Bank Carbon Pricing Dashboard via Our World in Data
# Tracks whether countries have carbon pricing instruments in place
OWID_CARBON_URL = "https://ourworldindata.org/grapher/carbon-tax-instruments.csv?v=1&csvType=full&useColumnShortNames=false"

print("Downloading carbon pricing data from Our World in Data...")
response = requests.get(OWID_CARBON_URL, timeout=30)
print(f"Status: {response.status_code}, Size: {len(response.content)/1024:.1f}KB")

carbon_raw = pd.read_csv(io.StringIO(response.text))
print(f"\nShape: {carbon_raw.shape}")
print(f"Columns: {list(carbon_raw.columns)}")
print(f"Years: {carbon_raw['Year'].min()} — {carbon_raw['Year'].max()}")
print(f"Entities: {carbon_raw['Entity'].nunique()}")
print(carbon_raw.head(3))

Status: 200, Size: 238.2KB

Shape: (7437, 4)
Columns: ['Entity', 'Code', 'Year', 'Covered by tax instrument in at least one sector']
Years: 1989 — 2025
Entities: 201
        Entity Code  Year Covered by tax instrument in at least one sector
0  Afghanistan  AFG  1989                                    No carbon tax
1  Afghanistan  AFG  1990                                    No carbon tax
2  Afghanistan  AFG  1991                                    No carbon tax


In [3]:
# Filter and rename columns
carbon = carbon_raw.copy()
carbon = carbon.rename(columns={
    'Entity': 'country_name',
    'Code':   'country_code',
    'Year':   'year',
    'Covered by tax instrument in at least one sector': 'wb_carbon_tax_coverage',
})

# Drop rows with no country code — regional aggregates
carbon = carbon[carbon['country_code'].notna() & (carbon['country_code'] != '')].copy()

# Filter to framework start year
carbon = carbon[carbon['year'] >= FRAMEWORK_START_YEAR].copy()
carbon = carbon.sort_values(['country_name', 'year']).reset_index(drop=True)

# Check unique values of carbon tax coverage
print(f"Shape: {carbon.shape}")
print(f"Years: {carbon['year'].min()} — {carbon['year'].max()}")
print(f"Countries: {carbon['country_name'].nunique()}")
print(f"\nUnique coverage values:")
print(carbon['wb_carbon_tax_coverage'].value_counts())

Shape: (7236, 4)
Years: 1990 — 2025
Countries: 201

Unique coverage values:
wb_carbon_tax_coverage
No carbon tax                                    6749
Has a carbon tax                                  469
Has a carbon tax only at a sub-national level      18
Name: count, dtype: int64


In [4]:
# Derive metadata from data — no hardcoding
latest_year = str(int(carbon['year'].max()))

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "wb_carbon_clean.csv")
carbon.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {carbon.shape}")

# Update download log
update_entry(
    "WB_CARBON",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=latest_year,
    local_filename="wb_carbon_clean.csv",
    latest_available_version=latest_year,
    notes="Carbon tax instrument coverage per country-year. "
          "Three values: No carbon tax / Has a carbon tax / Has a carbon tax only at a sub-national level. "
          "Downloaded via OWID. Original source: World Carbon Pricing Database (Dolphin & Merkle). "
          "Coverage: 1989-2025, 201 countries."
)
print_entry("WB_CARBON")

Written: /Users/boulanger/Documents/governance-framework/data/processed/wb_carbon_clean.csv
Shape: (7236, 4)
[download_log] Updated entry for WB_CARBON
  source_id: WB_CARBON
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2025
  local_filename: wb_carbon_clean.csv
  latest_available_version: 2025
  no_update_reason: nan
  notes: Carbon tax instrument coverage per country-year. Three values: No carbon tax / Has a carbon tax / Has a carbon tax only at a sub-national level. Downloaded via OWID. Original source: World Carbon Pricing Database (Dolphin & Merkle). Coverage: 1989-2025, 201 countries.
